# Visualize LitePT Validation Predictions

Notebook for inspecting `*_pred.npy` files saved by LitePT after fine-tuning. It supports both 2D masks and flat point predictions. For flat masks, the index strip is only a diagnostic view; the useful view is the point cloud scatter loaded from the matching `velodyne/<frame>.bin`.

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

plt.rcParams["figure.figsize"] = (16, 6)
plt.rcParams["axes.grid"] = False

# Change this if your fine-tune output is in another folder.
PROJECT_ROOT = Path("/home/a60116606/git_repo/noise_seg/pipline_v0")
RUN_DIR_CANDIDATES = [
    PROJECT_ROOT / "output" / "HL320-output_sam3_manual_104" / "fine_tune_100",
    PROJECT_ROOT / "output" / "HL320_output_sam3_manual_104" / "fine_tune_100",
    Path.cwd().parent / "output" / "HL320-output_sam3_manual_104" / "fine_tune_100",
    Path.cwd().parent / "output" / "HL320_output_sam3_manual_104" / "fine_tune_100",
]
RUN_DIR = next((path for path in RUN_DIR_CANDIDATES if path.exists()), RUN_DIR_CANDIDATES[0])
RESULT_DIR = RUN_DIR / "experiment" / "result"
TAXONOMY_PATH = RUN_DIR / "taxonomy.json"
RUN_MANIFEST_PATH = RUN_DIR / "run_manifest.json"

# Optional override. Leave None to read labeler_dir from run_manifest.json.
LABELER_DIR_OVERRIDE = None

# For flat 1D predictions only. If None, use 128 when divisible, otherwise 512.
INDEX_IMAGE_WIDTH = None

# "auto" usually works. Use "training" if *_pred.npy contains dense ids 0..N-1.
# Use "source" if *_pred.npy already contains original class ids from labels.xml.
PRED_ID_MODE = "auto"

print("RUN_DIR:", RUN_DIR)
print("RESULT_DIR:", RESULT_DIR)
print("TAXONOMY_PATH:", TAXONOMY_PATH)
print("RUN_MANIFEST_PATH:", RUN_MANIFEST_PATH)


In [ ]:
def read_json(path: Path) -> dict:
    if not path.is_file():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


taxonomy = read_json(TAXONOMY_PATH)
run_manifest = read_json(RUN_MANIFEST_PATH)

class_names = list(taxonomy.get("class_names") or run_manifest.get("taxonomy", {}).get("class_names") or [])
training_to_source = list(
    taxonomy.get("training_id_to_source_id")
    or run_manifest.get("taxonomy", {}).get("training_id_to_source_id")
    or range(len(class_names))
)
source_to_training = {
    int(source_id): int(training_id)
    for source_id, training_id in (taxonomy.get("source_id_to_training_id") or {}).items()
}
output_ignore_index = int(taxonomy.get("output_ignore_index", 255))
ignore_source_ids = {int(value) for value in taxonomy.get("ignore_source_ids", [255])}
source_id_to_name = {
    int(source_id): class_names[training_id]
    for training_id, source_id in enumerate(training_to_source)
    if training_id < len(class_names)
}
for ignore_id in ignore_source_ids | {output_ignore_index, 255}:
    source_id_to_name.setdefault(int(ignore_id), "ignore")

labeler_dir = Path(LABELER_DIR_OVERRIDE).expanduser() if LABELER_DIR_OVERRIDE else Path(run_manifest.get("labeler_dir", ""))
if str(labeler_dir) == ".":
    labeler_dir = Path("")

pred_files = sorted(RESULT_DIR.glob("*_pred.npy"))
print(f"classes: {len(class_names)}")
print("training_to_source:", training_to_source[:10], "..." if len(training_to_source) > 10 else "")
print("labeler_dir:", labeler_dir if str(labeler_dir) else "<not found>")
print(f"prediction files: {len(pred_files)}")
for path in pred_files[:10]:
    print(" ", path.name)


In [ ]:
def frame_id_from_pred(path: Path) -> str:
    name = path.stem
    return name[:-5] if name.endswith("_pred") else name


def stable_color(label_id: int) -> np.ndarray:
    label_id = int(label_id)
    if label_id in {0}:
        return np.array([35, 35, 35], dtype=np.uint8)
    if label_id in {255, output_ignore_index} | ignore_source_ids:
        return np.array([125, 125, 125], dtype=np.uint8)
    rng = np.random.default_rng((label_id * 1009 + 17) & 0xFFFFFFFF)
    return rng.integers(40, 240, size=3, dtype=np.uint8)


def unique_counts(labels: np.ndarray) -> list[tuple[int, int]]:
    values, counts = np.unique(labels.reshape(-1), return_counts=True)
    pairs = [(int(value), int(count)) for value, count in zip(values, counts)]
    return sorted(pairs, key=lambda item: item[1], reverse=True)


def infer_id_mode(raw: np.ndarray) -> str:
    if PRED_ID_MODE != "auto":
        return PRED_ID_MODE
    values = set(int(value) for value in np.unique(raw) if int(value) >= 0)
    non_ignore = values - ignore_source_ids - {255, output_ignore_index}
    if class_names and non_ignore and max(non_ignore) < len(class_names):
        return "training"
    return "source"


def prediction_to_source_ids(raw: np.ndarray, mode: str) -> np.ndarray:
    arr = np.asarray(raw)
    if mode == "source":
        return arr.astype(np.int32, copy=False)
    out = np.full(arr.shape, output_ignore_index, dtype=np.int32)
    for training_id, source_id in enumerate(training_to_source):
        out[arr == training_id] = int(source_id)
    out[arr < 0] = output_ignore_index
    return out


def colorize(labels: np.ndarray) -> np.ndarray:
    labels = np.asarray(labels)
    image = np.zeros(labels.shape + (3,), dtype=np.uint8)
    for label_id in np.unique(labels):
        image[labels == label_id] = stable_color(int(label_id))
    return image


def labels_to_index_image(labels: np.ndarray, width: int | None = None) -> np.ndarray:
    flat = np.asarray(labels).reshape(-1)
    if width is None:
        width = 128 if flat.size % 128 == 0 else 512
    height = int(math.ceil(flat.size / width))
    padded = np.full(height * width, output_ignore_index, dtype=np.int32)
    padded[: flat.size] = flat.astype(np.int32, copy=False)
    return padded.reshape(height, width)


def class_name(label_id: int) -> str:
    return source_id_to_name.get(int(label_id), f"id_{int(label_id)}")


def print_distribution(labels: np.ndarray, max_rows: int = 40) -> None:
    total = labels.size
    rows = ["| class id | name | points | percent |", "|---:|---|---:|---:|"]
    for label_id, count in unique_counts(labels)[:max_rows]:
        rows.append(f"| {label_id} | {class_name(label_id)} | {count} | {100.0 * count / total:.3f}% |")
    display(Markdown("\n".join(rows)))


In [ ]:
def load_points_for_frame(frame_id: str) -> np.ndarray | None:
    if not str(labeler_dir):
        return None
    path = labeler_dir / "velodyne" / f"{frame_id}.bin"
    if not path.is_file():
        print(f"No point cloud for {frame_id}: {path}")
        return None
    points = np.fromfile(path, dtype=np.float32)
    if points.size % 4 != 0:
        raise ValueError(f"{path} does not contain float32 XYZI data")
    return points.reshape(-1, 4)


def plot_prediction(pred_path: Path, *, scatter_size: float = 0.6, max_points: int | None = 300_000) -> None:
    raw = np.load(pred_path, allow_pickle=False)
    mode = infer_id_mode(raw)
    labels = prediction_to_source_ids(raw, mode)
    frame_id = frame_id_from_pred(pred_path)
    print(f"{pred_path.name}: raw shape={raw.shape}, dtype={raw.dtype}, id_mode={mode}, frame_id={frame_id}")
    print_distribution(labels)

    if labels.ndim == 2:
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        ax.imshow(colorize(labels), interpolation="nearest")
        ax.set_title(f"{frame_id} semantic prediction")
        ax.axis("off")
        plt.show()
        return

    index_image = labels_to_index_image(labels, width=INDEX_IMAGE_WIDTH)
    points = load_points_for_frame(frame_id)
    if points is None or points.shape[0] != labels.size:
        if points is not None:
            print(f"Point/prediction size mismatch: points={points.shape[0]}, labels={labels.size}")
        fig, ax = plt.subplots(1, 1, figsize=(16, 5))
        ax.imshow(colorize(index_image), interpolation="nearest", aspect="auto")
        ax.set_title(f"{frame_id} prediction by point index")
        ax.set_xlabel("point index modulo display width")
        ax.set_ylabel("point index block")
        plt.show()
        return

    colors = colorize(labels.reshape(-1)).reshape(-1, 3).astype(np.float32) / 255.0
    sample = np.arange(points.shape[0])
    if max_points is not None and points.shape[0] > max_points:
        rng = np.random.default_rng(42)
        sample = np.sort(rng.choice(sample, size=max_points, replace=False))

    pts = points[sample]
    cols = colors[sample]
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    axes[0].scatter(pts[:, 0], pts[:, 1], c=cols, s=scatter_size, linewidths=0)
    axes[0].set_title(f"{frame_id} XY")
    axes[0].set_xlabel("x")
    axes[0].set_ylabel("y")
    axes[0].axis("equal")

    axes[1].scatter(pts[:, 0], pts[:, 2], c=cols, s=scatter_size, linewidths=0)
    axes[1].set_title(f"{frame_id} XZ")
    axes[1].set_xlabel("x")
    axes[1].set_ylabel("z")
    axes[1].axis("equal")

    axes[2].imshow(colorize(index_image), interpolation="nearest", aspect="auto")
    axes[2].set_title("prediction by point index")
    axes[2].set_xlabel("point index modulo display width")
    axes[2].set_ylabel("point index block")
    plt.tight_layout()
    plt.show()


In [ ]:
# Pick one file. For your example this will select 000091_pred.npy if it exists.
if not pred_files:
    raise FileNotFoundError(f"No *_pred.npy files found in {RESULT_DIR}")

preferred = RESULT_DIR / "000091_pred.npy"
pred_path = preferred if preferred.is_file() else pred_files[0]
plot_prediction(pred_path)


In [ ]:
# Browse several predictions.
# Change start/count when needed.
start = 0
count = 5
for path in pred_files[start : start + count]:
    plot_prediction(path, scatter_size=0.4, max_points=150_000)


In [ ]:
def save_prediction_png(pred_path: Path, out_dir: Path | None = None) -> Path:
    raw = np.load(pred_path, allow_pickle=False)
    labels = prediction_to_source_ids(raw, infer_id_mode(raw))
    image_labels = labels if labels.ndim == 2 else labels_to_index_image(labels, width=INDEX_IMAGE_WIDTH)
    rgb = colorize(image_labels)
    out_dir = out_dir or (RUN_DIR / "prediction_previews")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{frame_id_from_pred(pred_path)}_pred_preview.png"
    plt.imsave(out_path, rgb)
    return out_path


# Save previews for all predictions.
# Uncomment if needed.
# saved = [save_prediction_png(path) for path in pred_files]
# print(f"saved {len(saved)} previews to {saved[0].parent if saved else ''}")
